In [ ]:
# Imports
import torch # type: ignore
import torch.nn as nn # type: ignore
import torch.nn.functional as f # type: ignore
from functorch.dim.reference import positional
from transformers import GPT2Tokenizer, GPT2Config # type: ignore

In [ ]:
# Attention class
class GPT2Attention(nn.Module):
    def __init__(self, config):
        super().__init__()
        max_positions = config.n_positions
        self.mask = (torch.tril(torch.ones(max_positions, max_positions, dtype=torch.bool))
                     .unsqueeze(0).unsqueeze(0))
        self.embed_dim = config.n_embd
        self.num_heads = config.n_head
        self.head_dim = self.embed_dim // self.num_heads
        self.split_size = self.embed_dim
        self.c_attn = nn.Linear(self.embed_dim, 3 * self.embed_dim)
        self.c_proj = nn.Linear(self.embed_dim, self.embed_dim)
        self.dropout = nn.Dropout(p=config.attn_pdrop)
    def _attn(self, query, key, value):
        # query, key, value: shape=(bach_size, num_heads, seq_len, head_dim)
        attn_weights = torch.matmul(query, key.transpose(-1, -2)) / torch.sqrt(self.head_dim)
        t = query.size(-2)
        casual_mask = self.mask[:, :, :t, :t].bool()
        attn_weights = torch.where(casual_mask, attn_weights, torch.tensor(-1e4))
        attn_weights = f.softmax(attn_weights, dim=-1)
        attn_weights = self.dropout(attn_weights)
        attn_output = torch.matmul(attn_weights, value)
        return attn_output
    def forward(self, x):
        # x: shape=(batch_size, seq_len, embed_dim)
        batch_size, sequence_len, embed_dim = x.size()
        query, key, value = self.c_attn(x).split(self.split_size, dim=-1)
        query = query.view(batch_size, sequence_len, self.num_heads, self.head_dim).permute(0, 2, 1, 3)
        key = key.view(batch_size, sequence_len, self.num_heads, self.head_dim).permute(0, 2, 1, 3)
        value = value.view(batch_size, sequence_len, self.num_heads, self.head_dim).permute(0, 2, 1, 3)
        # query, key, value: shape=(bach_size, num_heads, seq_len, head_dim)
        attn_output = self._attn(query, key, value)
        attn_output = attn_output.permute(0, 2, 1, 3).view(batch_size, sequence_len, embed_dim)
        # attn_output: shape=(batch_size, seq_len, embed_dim)
        attn_output = self.c_proj(attn_output)
        attn_output = self.dropout(attn_output)
        return attn_output

In [ ]:
# MLP class for transformer
class GPT2MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        embed_dim = config.n_embd
        self.mlp = nn.Sequential(nn.Linear(embed_dim, 4 * embed_dim),
                                 nn.GELU(),
                                 nn.Linear(4 * embed_dim, embed_dim),
                                 nn.Dropout(p=config.resid_pdrop))
    def forward(self, x):
        return self.mlp(x)

In [ ]:
# Layer block
class GPT2Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        embed_dim = config.n_embd
        self.ln_1 = nn.LayerNorm(embed_dim)
        self.ln_2 = nn.LayerNorm(embed_dim)
        self.attn = GPT2Attention(config)
        self.mlp = GPT2MLP(config)
    def forward(self, hidden_state):
        residual = hidden_state
        hidden_state = self.ln_1(hidden_state)
        attn_output = self.attn(hidden_state)
        hidden_state = attn_output + residual
        residual = hidden_state
        hidden_state = self.ln_2(hidden_state)
        feed_forward_hidden_state = self.mlp(hidden_state)
        hidden_state = feed_forward_hidden_state + residual
        return hidden_state

In [7]:
# Model class
class GPT2Model(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.embed_dim = config.n_embd
        self.vocab_size = config.vocab_size
        self.n_positions = config.n_positions
        self.resid_pdrop = config.resid_pdrop
        self.n_layers = config.n_layers
        self.wte = nn.Embedding(self.vocab_size, self.embed_dim)
        self.wpe = nn.Embedding(self.n_positions, self.embed_dim)
        self.dropout = nn.Dropout(self.resid_pdrop)
        self.blocks = nn.ModuleList([GPT2Block(config) for _ in range(self.n_layers)])
        self.ln_f = nn.LayerNorm(self.embed_dim)
    def forward(self, input_ids=None, position_ids=None):
        # input_ids: shape=(batch_size, max_seq_len)
        batch_size = input_ids.size(0)
        max_length = input_ids.size(1)
        device = input_ids.device
        if not position_ids:
            position_ids = torch.arange(0, max_length, dtype=torch.long, device=device).unsqueeze(0)
        input_embeds = self.wte(input_ids)
        position_embeds = self.wpe(position_ids)
        hidden_state = input_embeds + position_embeds
        hidden_state = self.dropout(hidden_state)
        for block in self.blocks:
            hidden_state = block(hidden_state)
        hidden_state = self.ln_f(hidden_state)
        return hidden_state

In [10]:
# Language model classifier
class GPT2LMHead(nn.Module):
    def __init__(self, config, tokenizer):
        super().__init__()
        self.embed_dim = config.n_embd
        self.vocab_size = config.vocab_size
        self.transformer = GPT2Model(config)
        self.LMHead = nn.Linear(self.embed_dim, config.vocab_size, bias=False)
        self.xe = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token)
    def forward(self, input_ids=None, position_ids=None, labels=None):
        hidden_state = self.transformer(input_ids)
        lm_logits = self.LMHead(hidden_state)
        loss = None
        if labels is not None:
            shift_logits = lm_logits[:, :-1, :]
            shift_labels = labels[:, 1:, :]
            loss = self.xe(shift_logits.view(-1, self.vocab_size),
                           shift_labels.view(-1))
        return lm_logits, loss